# 04 — Target Size Analysis

Focused analysis of thermal-drone bounding-box sizes. The goal is to quantify how many targets are genuinely tiny, especially after the ×4 downsampling used by the SR experiment.

This notebook uses the corrected YOLO annotations and reports **target instances**, not images. No SR metric or detector result is evaluated here.


## 1. Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_REPO = 'https://github.com/shelly-serafimovich/target-aware-infrared-sr.git'
project_dir = Path('/content/target-aware-infrared-sr')
if not project_dir.exists():
    subprocess.run(['git','clone',PROJECT_REPO,str(project_dir)],check=True)
else:
    subprocess.run(['git','pull','--ff-only'],cwd=project_dir,check=True)

DATASET_ROOT = Path('/content/drive/MyDrive/thermal-drone-dataset')
ANN_ROOT = project_dir / 'annotations' / 'corrected_yolo'
SCALE = 4
splits = ['train','valid','test']
print('Annotations:', ANN_ROOT)


## 2. Build one row per target

YOLO boxes are normalized. We recover pixel width/height using the source image dimensions. `max_dim_lr` is the largest bbox dimension after ideal ×4 spatial downsampling.


In [ ]:
import zipfile

LOCAL_IMAGES = Path('/content/target_size_images')
LOCAL_IMAGES.mkdir(exist_ok=True)
exts = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}

def image_dir_for_split(split):
    src = DATASET_ROOT / split
    direct = [p for p in src.iterdir() if p.is_file() and p.suffix.lower() in exts]
    if direct:
        return src
    zips = list(src.glob('*.zip'))
    if len(zips) != 1:
        raise FileNotFoundError(f'Expected one ZIP for {split}, found {len(zips)}')
    out = LOCAL_IMAGES / split
    marker = out / '.complete'
    if not marker.exists():
        if out.exists():
            import shutil; shutil.rmtree(out)
        out.mkdir(parents=True)
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(out)
        marker.touch()
    candidates = [out] + [p for p in out.rglob('*') if p.is_dir()]
    for p in candidates:
        if any(x.is_file() and x.suffix.lower() in exts for x in p.iterdir()):
            return p
    raise FileNotFoundError(f'No images found for {split}')

rows=[]
for split in splits:
    img_dir=image_dir_for_split(split)
    image_map={p.stem:p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in exts}
    label_dir=ANN_ROOT/split/'labels'
    for lp in sorted(label_dir.glob('*.txt')):
        lines=[x.strip() for x in lp.read_text().splitlines() if x.strip()]
        if not lines:
            continue
        img_path=image_map.get(lp.stem)
        if img_path is None:
            raise FileNotFoundError(f'Image missing for {split}/{lp.stem}')
        with Image.open(img_path) as im:
            W,H=im.size
        for target_idx,line in enumerate(lines):
            p=line.split()
            if len(p)<5:
                continue
            _,xc,yc,bw,bh=map(float,p[:5])
            w=bw*W; h=bh*H
            rows.append({'split':split,'image':lp.stem,'target_idx':target_idx,
                         'image_w':W,'image_h':H,'width_hr':w,'height_hr':h,
                         'max_dim_hr':max(w,h),'diagonal_hr':np.hypot(w,h),'area_hr':w*h,
                         'width_lr':w/SCALE,'height_lr':h/SCALE,
                         'max_dim_lr':max(w,h)/SCALE,'diagonal_lr':np.hypot(w,h)/SCALE,
                         'area_lr':(w*h)/(SCALE**2)})

df=pd.DataFrame(rows)
print(f'Target instances: {len(df):,}')
display(df.head())


## 3. How many targets are tiny?

We report explicit thresholds instead of relying on a vague definition of “small”. The main variable is `max_dim_lr`: the larger side of the target bbox in the LR image.


In [ ]:
bins=[0,2,4,8,16,np.inf]
labels=['≤2 px','2–4 px','4–8 px','8–16 px','>16 px']
df['lr_size_bin']=pd.cut(df['max_dim_lr'],bins=bins,labels=labels,include_lowest=True,right=True)

counts=df['lr_size_bin'].value_counts(sort=False)
table=pd.DataFrame({'targets':counts,'percent':100*counts/len(df)})
display(table.round({'percent':2}))

for t in [2,4,8,16]:
    n=int((df.max_dim_lr<=t).sum())
    print(f'max_dim_lr ≤ {t:>2}px: {n:>5,} / {len(df):,} = {100*n/len(df):6.2f}%')


## 4. Distribution by split

This checks whether the tiny-target population is concentrated in one split or is represented across train/validation/test.


In [ ]:
split_table=pd.crosstab(df['split'],df['lr_size_bin'])
split_pct=100*split_table.div(split_table.sum(axis=1),axis=0)
display(split_table)
display(split_pct.round(2))


## 5. Distribution plots


In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df['max_dim_lr'],bins=50)
plt.xlabel('Target max bbox dimension in LR (pixels)')
plt.ylabel('Number of target instances')
plt.title('Thermal-drone target size after ×4 downsampling')
plt.xlim(0, min(40, df['max_dim_lr'].quantile(0.99)))
plt.show()

plt.figure(figsize=(8,5))
table['percent'].plot(kind='bar')
plt.xlabel('LR target-size bin')
plt.ylabel('Targets (%)')
plt.title('Target-size composition of the dataset')
plt.xticks(rotation=0)
plt.show()


## 6. Compact summary

Use these values when deciding whether statements such as “there are almost no small drones” are supported by the corrected dataset.


In [ ]:
summary={
    'n_targets':int(len(df)),
    'median_max_dim_hr':float(df.max_dim_hr.median()),
    'median_max_dim_lr':float(df.max_dim_lr.median()),
    'pct_lr_le_2':float(100*(df.max_dim_lr<=2).mean()),
    'pct_lr_le_4':float(100*(df.max_dim_lr<=4).mean()),
    'pct_lr_le_8':float(100*(df.max_dim_lr<=8).mean()),
    'pct_lr_le_16':float(100*(df.max_dim_lr<=16).mean()),
}
for k,v in summary.items():
    print(f'{k}: {v}')


## 7. Pixel-area buckets and 9×9 window coverage

Here we bucket targets by **LR bounding-box area in pixels**. We also report two interpretations of a 9×9 window: (1) bbox area ≤81 px², and (2) the stricter/geometrically correct condition for fitting fully inside a 9×9 window: width ≤9 px **and** height ≤9 px.


In [ ]:
# Integer pixel-area buckets (ceil so a fractional bbox is not understated)
df['area_lr_px_ceil'] = np.ceil(df['area_lr']).astype(int)

area_counts = (df.groupby('area_lr_px_ceil')
                 .agg(targets=('image','size'), images=('image','nunique'))
                 .reset_index()
                 .rename(columns={'area_lr_px_ceil':'pixel_area_bucket'}))
area_counts['target_percent'] = 100 * area_counts['targets'] / len(df)
display(area_counts)

# 9×9 checks
by_area = df['area_lr'] <= 81
fits_9x9 = (df['width_lr'] <= 9) & (df['height_lr'] <= 9)

def report(mask, name):
    sub = df[mask]
    print(f'{name}:')
    print(f'  targets: {len(sub):,} / {len(df):,} = {100*len(sub)/len(df):.2f}%')
    print(f'  unique images: {sub.image.nunique():,} / {df.image.nunique():,} = {100*sub.image.nunique()/df.image.nunique():.2f}%')

report(by_area, 'LR bbox area ≤ 81 px²')
report(fits_9x9, 'LR bbox fits fully inside 9×9 (width≤9 AND height≤9)')


## 8. Visual examples from each LR target-size group

Show several examples from each `max_dim_lr` bucket. Each panel displays the **full HR source image** with the selected target bbox highlighted and reports the corresponding LR bbox size. This is for visual sanity-checking of what each numerical size group actually looks like.


In [ ]:
from matplotlib.patches import Rectangle

N_EXAMPLES = 6
rng = np.random.default_rng(12345)

# Rebuild image lookup once for visualization.
split_image_maps = {}
for split in splits:
    d = image_dir_for_split(split)
    split_image_maps[split] = {p.stem:p for p in d.iterdir() if p.is_file() and p.suffix.lower() in exts}

def get_target_xywh(row):
    lp = ANN_ROOT / row['split'] / 'labels' / f"{row['image']}.txt"
    lines = [x.strip() for x in lp.read_text().splitlines() if x.strip()]
    p = lines[int(row['target_idx'])].split()
    _, xc, yc, bw, bh = map(float, p[:5])
    W, H = row['image_w'], row['image_h']
    return (xc-bw/2)*W, (yc-bh/2)*H, bw*W, bh*H

for group in labels:
    group_df = df[df['lr_size_bin'] == group]
    n = min(N_EXAMPLES, len(group_df))
    print(f'\n{group}: {len(group_df):,} targets — showing {n}')
    if n == 0:
        continue
    sample = group_df.sample(n=n, random_state=12345)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    axes = np.atleast_1d(axes)
    for ax, (_, row) in zip(axes, sample.iterrows()):
        img_path = split_image_maps[row['split']][row['image']]
        img = Image.open(img_path).convert('L')
        ax.imshow(img, cmap='gray')
        x, y, w, h = get_target_xywh(row)
        ax.add_patch(Rectangle((x,y),w,h,fill=False,linewidth=2))
        ax.set_title(f"{group}\nLR {row['width_lr']:.1f}×{row['height_lr']:.1f}px\nHR {row['width_hr']:.1f}×{row['height_hr']:.1f}px")
        ax.axis('off')
    plt.tight_layout()
    plt.show()
